# Subgraphs Pattern

**Pattern**: Use a compiled graph as a node within another graph for modularity and reusability.

**Example**: Document Processing Pipeline with specialized processors

**Workflow**:
```
                              ┌→ [Research Agent Subgraph] ─┐
START → [Router] ────────────┼→ [Summary Agent Subgraph]  ─┼→ [Format Output] → END
                              └→ [QA Agent Subgraph]       ─┘
```

**Key Concepts**:
- Subgraph = compiled graph used as a node
- Pattern 1: Wrapper function (different schemas)
- Pattern 2: Direct node (shared schemas)

## Setup

In [ ]:
from typing import TypedDict, Literal, Annotated
import operator

from langgraph.graph import StateGraph, START, END
from langchain_aws import ChatBedrockConverse

## Initialize LLM

In [ ]:
llm = ChatBedrockConverse(
    model="us.anthropic.claude-sonnet-4-6",
    temperature=0,
)

---
# Part 1: Pattern 1 - Wrapper Function (Different Schemas)

Use when parent and subgraph have **different state schemas**.

## Define Subgraph State (Different from Parent)

In [ ]:
# Subgraph has its own state schema
class SummarizerState(TypedDict):
    text_to_summarize: str
    summary_result: str
    word_count: int

## Build Summarizer Subgraph

In [ ]:
def analyze_text(state: SummarizerState) -> dict:
    """Count words in the text."""
    word_count = len(state["text_to_summarize"].split())
    return {"word_count": word_count}

def create_summary(state: SummarizerState) -> dict:
    """Generate summary using LLM."""
    prompt = f"""Summarize this text in 2-3 sentences:

{state['text_to_summarize']}

Summary:"""
    response = llm.invoke(prompt)
    return {"summary_result": response.content}

# Build subgraph
summarizer_builder = StateGraph(SummarizerState)
summarizer_builder.add_node("analyze", analyze_text)
summarizer_builder.add_node("summarize", create_summary)
summarizer_builder.add_edge(START, "analyze")
summarizer_builder.add_edge("analyze", "summarize")
summarizer_builder.add_edge("summarize", END)

# Compile subgraph
summarizer_subgraph = summarizer_builder.compile()

## Test Subgraph in Isolation

In [ ]:
# Test the subgraph standalone
test_result = summarizer_subgraph.invoke({
    "text_to_summarize": "LangGraph is a framework for building stateful AI applications. It uses a graph-based approach where nodes represent tasks and edges define the flow. State is shared across all nodes and persists throughout execution."
})

print(f"Word count: {test_result['word_count']}")
print(f"Summary: {test_result['summary_result']}")

## Define Parent State (Different Schema)

In [ ]:
# Parent has different field names
class DocumentState(TypedDict):
    document_content: str      # Different from 'text_to_summarize'
    document_summary: str      # Different from 'summary_result'
    processing_status: str

## Create Wrapper Function

Transforms state between parent and subgraph schemas.

In [ ]:
def call_summarizer_subgraph(state: DocumentState) -> dict:
    """
    Wrapper function that:
    1. Transforms parent state -> subgraph input
    2. Invokes subgraph
    3. Transforms subgraph output -> parent state update
    """
    # Transform: parent -> subgraph input
    subgraph_input = {
        "text_to_summarize": state["document_content"]
    }
    
    # Invoke subgraph
    subgraph_output = summarizer_subgraph.invoke(subgraph_input)
    
    # Transform: subgraph output -> parent state update
    return {
        "document_summary": subgraph_output["summary_result"],
        "processing_status": f"Summarized ({subgraph_output['word_count']} words)"
    }

## Build Parent Graph with Wrapper

In [ ]:
def validate_document(state: DocumentState) -> dict:
    """Initial validation step."""
    if len(state["document_content"]) < 10:
        return {"processing_status": "Error: Document too short"}
    return {"processing_status": "Validated"}

def format_output(state: DocumentState) -> dict:
    """Final formatting step."""
    return {"processing_status": state["processing_status"] + " | Complete"}

# Build parent graph
parent_builder = StateGraph(DocumentState)
parent_builder.add_node("validate", validate_document)
parent_builder.add_node("summarize", call_summarizer_subgraph)  # Wrapper as node
parent_builder.add_node("format", format_output)

parent_builder.add_edge(START, "validate")
parent_builder.add_edge("validate", "summarize")
parent_builder.add_edge("summarize", "format")
parent_builder.add_edge("format", END)

parent_graph = parent_builder.compile()

## Visualize Parent Graph

In [ ]:
from IPython.display import Image, display

display(Image(parent_graph.get_graph().draw_mermaid_png()))

## Test Pattern 1: Wrapper Function

In [ ]:
result = parent_graph.invoke({
    "document_content": """Artificial Intelligence has transformed numerous industries over the past decade. 
    Machine learning algorithms now power recommendation systems, autonomous vehicles, and medical diagnostics. 
    Deep learning, a subset of ML, has achieved remarkable results in image recognition and natural language processing. 
    Companies worldwide are investing billions in AI research and development.""",
    "document_summary": "",
    "processing_status": ""
})

print("=" * 60)
print("PATTERN 1: Wrapper Function (Different Schemas)")
print("=" * 60)
print(f"\nStatus: {result['processing_status']}")
print(f"\nSummary:\n{result['document_summary']}")

---
# Part 2: Pattern 2 - Direct Node (Shared Schema)

Use when parent and subgraph **share the same state keys**.

## Define Shared State Schema

In [ ]:
# Both parent and subgraph use this schema
class SharedState(TypedDict):
    query: str
    context: str
    response: str
    confidence: float

## Build Research Subgraph (Shared Schema)

In [ ]:
def gather_context(state: SharedState) -> dict:
    """Simulate gathering research context."""
    # In real app, this would search databases, APIs, etc.
    prompt = f"""Provide 3 key facts about: {state['query']}

Format as bullet points."""
    response = llm.invoke(prompt)
    return {"context": response.content}

def assess_confidence(state: SharedState) -> dict:
    """Assess confidence in gathered context."""
    # Simple heuristic: more content = higher confidence
    context_length = len(state["context"])
    confidence = min(context_length / 500, 1.0)
    return {"confidence": confidence}

# Build research subgraph
research_builder = StateGraph(SharedState)
research_builder.add_node("gather", gather_context)
research_builder.add_node("assess", assess_confidence)
research_builder.add_edge(START, "gather")
research_builder.add_edge("gather", "assess")
research_builder.add_edge("assess", END)

research_subgraph = research_builder.compile()

## Build Answer Subgraph (Shared Schema)

In [ ]:
def generate_answer(state: SharedState) -> dict:
    """Generate answer based on context."""
    prompt = f"""Based on this context, answer the question.

Context:
{state['context']}

Question: {state['query']}

Answer:"""
    response = llm.invoke(prompt)
    return {"response": response.content}

# Build answer subgraph
answer_builder = StateGraph(SharedState)
answer_builder.add_node("answer", generate_answer)
answer_builder.add_edge(START, "answer")
answer_builder.add_edge("answer", END)

answer_subgraph = answer_builder.compile()

## Build Parent Graph with Direct Subgraph Nodes

In [ ]:
def initialize_query(state: SharedState) -> dict:
    """Initialize the query processing."""
    return {"context": "", "response": "", "confidence": 0.0}

# Build parent graph
qa_builder = StateGraph(SharedState)

# Add nodes
qa_builder.add_node("init", initialize_query)
qa_builder.add_node("research", research_subgraph)  # Direct subgraph (no wrapper!)
qa_builder.add_node("answer", answer_subgraph)      # Direct subgraph (no wrapper!)

# Add edges
qa_builder.add_edge(START, "init")
qa_builder.add_edge("init", "research")
qa_builder.add_edge("research", "answer")
qa_builder.add_edge("answer", END)

qa_graph = qa_builder.compile()

## Visualize QA Graph

In [ ]:
display(Image(qa_graph.get_graph().draw_mermaid_png()))

## Test Pattern 2: Direct Node

In [ ]:
result = qa_graph.invoke({
    "query": "What are the benefits of using LangGraph for AI applications?"
})

print("=" * 60)
print("PATTERN 2: Direct Node (Shared Schema)")
print("=" * 60)
print(f"\nQuery: {result['query']}")
print(f"\nConfidence: {result['confidence']:.2f}")
print(f"\nContext:\n{result['context']}")
print(f"\nResponse:\n{result['response']}")

---
# Part 3: Multi-Agent System with Routing

Combine subgraphs with conditional routing.

## Define Multi-Agent State

In [ ]:
class AgentState(TypedDict):
    user_request: str
    request_type: str      # research, summary, qa
    agent_output: str
    final_response: str

## Build Specialized Agent Subgraphs

In [ ]:
# Research Agent
def research_task(state: AgentState) -> dict:
    prompt = f"""You are a research agent. Provide detailed research on:
{state['user_request']}

Include facts, statistics, and sources where possible."""
    response = llm.invoke(prompt)
    return {"agent_output": response.content}

research_agent_builder = StateGraph(AgentState)
research_agent_builder.add_node("research", research_task)
research_agent_builder.add_edge(START, "research")
research_agent_builder.add_edge("research", END)
research_agent = research_agent_builder.compile()

# Summary Agent
def summary_task(state: AgentState) -> dict:
    prompt = f"""You are a summary agent. Create a concise summary of:
{state['user_request']}

Keep it under 100 words."""
    response = llm.invoke(prompt)
    return {"agent_output": response.content}

summary_agent_builder = StateGraph(AgentState)
summary_agent_builder.add_node("summarize", summary_task)
summary_agent_builder.add_edge(START, "summarize")
summary_agent_builder.add_edge("summarize", END)
summary_agent = summary_agent_builder.compile()

# QA Agent
def qa_task(state: AgentState) -> dict:
    prompt = f"""You are a Q&A agent. Answer this question directly:
{state['user_request']}

Be concise and accurate."""
    response = llm.invoke(prompt)
    return {"agent_output": response.content}

qa_agent_builder = StateGraph(AgentState)
qa_agent_builder.add_node("qa", qa_task)
qa_agent_builder.add_edge(START, "qa")
qa_agent_builder.add_edge("qa", END)
qa_agent = qa_agent_builder.compile()

## Build Orchestrator with Routing

In [ ]:
def classify_request(state: AgentState) -> dict:
    """Classify the type of request."""
    request = state["user_request"].lower()
    
    if any(word in request for word in ["research", "find out", "investigate", "explore"]):
        return {"request_type": "research"}
    elif any(word in request for word in ["summarize", "summary", "brief", "short"]):
        return {"request_type": "summary"}
    else:
        return {"request_type": "qa"}

def route_to_agent(state: AgentState) -> Literal["research_agent", "summary_agent", "qa_agent"]:
    """Route to appropriate agent based on request type."""
    request_type = state["request_type"]
    if request_type == "research":
        return "research_agent"
    elif request_type == "summary":
        return "summary_agent"
    else:
        return "qa_agent"

def format_final_response(state: AgentState) -> dict:
    """Format the final response."""
    response = f"""[{state['request_type'].upper()} AGENT]

{state['agent_output']}"""
    return {"final_response": response}

In [ ]:
# Build orchestrator
orchestrator_builder = StateGraph(AgentState)

# Add nodes
orchestrator_builder.add_node("classify", classify_request)
orchestrator_builder.add_node("research_agent", research_agent)  # Subgraph
orchestrator_builder.add_node("summary_agent", summary_agent)    # Subgraph
orchestrator_builder.add_node("qa_agent", qa_agent)              # Subgraph
orchestrator_builder.add_node("format", format_final_response)

# Add edges
orchestrator_builder.add_edge(START, "classify")

# Conditional routing to agents
orchestrator_builder.add_conditional_edges(
    "classify",
    route_to_agent,
    {
        "research_agent": "research_agent",
        "summary_agent": "summary_agent",
        "qa_agent": "qa_agent"
    }
)

# All agents lead to format
orchestrator_builder.add_edge("research_agent", "format")
orchestrator_builder.add_edge("summary_agent", "format")
orchestrator_builder.add_edge("qa_agent", "format")
orchestrator_builder.add_edge("format", END)

orchestrator = orchestrator_builder.compile()

## Visualize Multi-Agent Orchestrator

In [ ]:
display(Image(orchestrator.get_graph().draw_mermaid_png()))

## Test Multi-Agent System

In [ ]:
# Test 1: Research request
result = orchestrator.invoke({
    "user_request": "Research the latest developments in quantum computing"
})

print("=" * 60)
print("TEST 1: Research Request")
print("=" * 60)
print(result["final_response"])

In [ ]:
# Test 2: Summary request
result = orchestrator.invoke({
    "user_request": "Give me a brief summary of machine learning applications in healthcare"
})

print("=" * 60)
print("TEST 2: Summary Request")
print("=" * 60)
print(result["final_response"])

In [ ]:
# Test 3: QA request
result = orchestrator.invoke({
    "user_request": "What is the capital of France?"
})

print("=" * 60)
print("TEST 3: QA Request")
print("=" * 60)
print(result["final_response"])

## Stream with Subgraph Visibility

In [ ]:
print("=" * 60)
print("STREAMING WITH SUBGRAPH VISIBILITY")
print("=" * 60)

for chunk in orchestrator.stream(
    {"user_request": "Research AI ethics"},
    subgraphs=True
):
    # chunk is a tuple: (namespace, data)
    if isinstance(chunk, tuple):
        ns, data = chunk
        ns_str = "ROOT" if not ns else " > ".join(ns)
        for node, output in data.items():
            print(f"\n[{ns_str}] Node: {node}")
            if "agent_output" in output:
                print(f"   Output: {output['agent_output'][:100]}...")
            elif "request_type" in output:
                print(f"   Routed to: {output['request_type']}")

## Key Takeaways

1. **Pattern 1 (Wrapper)**: Use when schemas differ - wrapper transforms state bidirectionally
2. **Pattern 2 (Direct)**: Use when schemas match - add subgraph directly as node
3. **Subgraphs enable modularity**: Build and test agents independently
4. **Combine with routing**: Use conditional edges to select which subgraph to invoke
5. **Stream with `subgraphs=True`**: See outputs from nested subgraphs

### When to Use Each Pattern

| Scenario | Pattern |
|----------|--------|
| Different teams, different schemas | Wrapper function |
| Multi-agent with shared messages | Direct node |
| Reusing existing graph | Wrapper (adapt interface) |
| Same state keys throughout | Direct node |